Autor: David Díaz Paz y Puente

### Universidad de Monterrey
# Proyecto Final – Unidad 3: Aprendizaje no supervisado



In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

# =========================
# 1. Cargar archivos
# =========================

data_dir = Path("data_bases")

expr_path = data_dir / "TCGA-UCS.star_tpm.tsv.gz"
clinical_path = data_dir / "TCGA-UCS.clinical.tsv.gz"
survival_path = data_dir / "TCGA-UCS.survival.tsv.gz"

expr = pd.read_csv(expr_path, sep="\t", compression="gzip", index_col=0)
clinical = pd.read_csv(clinical_path, sep="\t", compression="gzip")
survival = pd.read_csv(survival_path, sep="\t", compression="gzip")

print("Expresión génica:", expr.shape)
print("Clinical:", clinical.shape)
print("Survival:", survival.shape)

display(expr.head())
display(clinical.head())
display(survival.head())

Expresión génica: (60660, 57)
Clinical: (57, 64)
Survival: (55, 4)


,TCGA-NF-A4X2-01A,TCGA-N5-A4RA-01A,TCGA-NG-A4VU-01A,TCGA-N5-A4RS-01A,TCGA-NA-A5I1-01A,TCGA-N8-A4PL-01A,TCGA-ND-A4W6-01A,TCGA-NG-A4VW-01A,TCGA-ND-A4WF-01A,TCGA-NA-A4QW-01A,...,TCGA-N6-A4V9-01A,TCGA-N9-A4Q1-01A,TCGA-N9-A4Q4-01A,TCGA-N5-A4RD-01A,TCGA-NF-A5CP-01A,TCGA-N8-A4PP-01A,TCGA-N5-A59E-01A,TCGA-N8-A4PO-01A,TCGA-QM-A5NM-01A,TCGA-N9-A4Q7-01A
Ensembl_ID,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15,5.664514,4.757637,4.283707,6.219244,6.763791,6.190398,5.474757,6.408695,6.361642,5.497002,...,7.114856,5.667685,5.061042,5.798108,5.769439,5.352932,6.118613,6.335213,5.479289,4.196827
ENSG00000000005.6,2.962623,0.311387,0.681224,4.098554,5.207643,6.388791,3.431302,4.094050,10.450645,6.009085,...,1.523060,1.521604,2.017779,2.720607,1.001874,4.521566,6.822915,1.413757,3.611668,0.106750
ENSG00000000419.13,6.889567,7.075702,7.480493,6.324899,7.232189,7.191428,6.807755,7.035680,7.018747,7.198450,...,6.997765,6.543337,7.983903,7.810991,7.181593,7.469926,6.873561,7.614342,6.648027,7.108394
ENSG00000000457.14,3.142413,2.476589,2.908044,2.640228,3.062484,3.209734,1.687822,4.292465,2.653152,2.934309,...,3.103330,2.218719,2.708607,2.636520,2.811450,2.923929,2.650581,3.232016,2.265437,1.941181
ENSG00000000460.17,3.731303,2.829789,3.338439,1.371503,2.634663,3.814879,2.206924,3.540027,2.643764,3.189255,...,2.787474,2.223608,3.306977,3.570766,2.777977,3.227418,2.477781,3.528809,2.323024,2.345198


,sample,id,disease_type,case_id,submitter_id,primary_site,alcohol_history.exposures,race.demographic,gender.demographic,ethnicity.demographic,...,sample_id.samples,sample_type.samples,days_to_collection.samples,initial_weight.samples,preservation_method.samples,pathology_report_uuid.samples,oct_embedded.samples,specimen_type.samples,is_ffpe.samples,tissue_type.samples
0,TCGA-N5-A4RM-01A,254f6921-82ac-4fe1-b1b4-ace94c420d05,Complex Mixed and Stromal Neoplasms,254f6921-82ac-4fe1-b1b4-ace94c420d05,TCGA-N5-A4RM,"Uterus, NOS",Not Reported,white,female,not hispanic or latino,...,3165ff33-e042-4c34-8bc4-ab2358f16803,Primary Tumor,830.0,250.0,OCT,C188B8E8-B564-45A6-B356-B4D7C64B8DA5,True,Solid Tissue,False,Tumor
1,TCGA-N9-A4Q7-01A,9f15bf13-11d2-400f-b6f9-c8be0a57c763,Complex Mixed and Stromal Neoplasms,9f15bf13-11d2-400f-b6f9-c8be0a57c763,TCGA-N9-A4Q7,"Uterus, NOS",Not Reported,black or african american,female,not reported,...,5ccfd055-ec96-4976-916d-a4d2becb8f6f,Primary Tumor,438.0,420.0,Unknown,0E5E42BD-AB48-4889-BFC8-1D6E6657CB85,False,Solid Tissue,False,Tumor
2,TCGA-N6-A4VD-01A,14213209-2217-4812-9a19-d9b2b6718467,Complex Mixed and Stromal Neoplasms,14213209-2217-4812-9a19-d9b2b6718467,TCGA-N6-A4VD,"Uterus, NOS",Not Reported,white,female,not hispanic or latino,...,56c9a562-e195-41da-9e39-8c19dca82b97,Primary Tumor,1297.0,400.0,OCT,B1D9BFB6-3943-457E-ACBC-89504A5CD7E0,True,Solid Tissue,False,Tumor
3,TCGA-N5-A59F-01A,158fd4c2-68dc-4ca4-8130-d5589c80d825,Complex Mixed and Stromal Neoplasms,158fd4c2-68dc-4ca4-8130-d5589c80d825,TCGA-N5-A59F,"Uterus, NOS",Not Reported,black or african american,female,not reported,...,9dcc425f-677e-4143-82cd-dc213d00a217,Primary Tumor,238.0,170.0,OCT,5666CA72-D8E1-4781-8AE7-2321BDB57323,True,Solid Tissue,False,Tumor
4,TCGA-N5-A4RT-01A,1791e250-70ac-439c-828b-15ba811935cc,Complex Mixed and Stromal Neoplasms,1791e250-70ac-439c-828b-15ba811935cc,TCGA-N5-A4RT,"Uterus, NOS",Not Reported,white,female,not reported,...,d43afdad-c03b-47c7-8727-570f410dbc96,Primary Tumor,1977.0,300.0,OCT,AD77201F-0CF7-4C9B-ABDE-94F4FCAF3C0C,True,Solid Tissue,False,Tumor


,sample,OS.time,OS,_PATIENT
0,TCGA-NF-A5CP-01A,1.0,1,TCGA-NF-A5CP
1,TCGA-N7-A4Y5-01A,8.0,1,TCGA-N7-A4Y5
2,TCGA-NF-A4X2-01A,81.0,0,TCGA-NF-A4X2
3,TCGA-N6-A4VD-01A,112.0,1,TCGA-N6-A4VD
4,TCGA-NA-A4QY-01A,114.0,1,TCGA-NA-A4QY


Eso significa que hay 60,660 genes y 57 muestras/pacientes en expresión génica. El archivo clínico tiene información para las 57 muestras, pero survival solo tiene 55, entonces hay 2 pacientes sin datos de supervivencia. Esto no es problema debido a que el clustering se hace con los 57 pacientes usando solo expresión génica, y la supervivencia se analiza después únicamente en los pacientes que tengan datos disponibles. Esto va alineado con la instrucción del PDF: el clustering debe realizarse solo con expresión génica, mientras que los datos clínicos se usan después para interpretar los clusters

In [ ]:
# =========================
# 2. Transponer expresión génica
# =========================

X = expr.T
X = X.apply(pd.to_numeric, errors="coerce")

print("Matriz para machine learning:", X.shape)
display(X.head())


# =========================
# 3. Verificar coincidencia de muestras
# =========================

expr_samples = set(X.index)
clinical_samples = set(clinical["sample"])
survival_samples = set(survival["sample"])

print("Muestras en expresión:", len(expr_samples))
print("Muestras en clinical:", len(clinical_samples))
print("Muestras en survival:", len(survival_samples))

print("\nMuestras de expresión que no están en clinical:")
print(expr_samples - clinical_samples)

print("\nMuestras de clinical que no están en expresión:")
print(clinical_samples - expr_samples)

print("\nMuestras de expresión que no están en survival:")
print(expr_samples - survival_samples)


# =========================
# 4. Crear tabla base de muestras
# =========================

samples = pd.DataFrame({
    "sample": X.index
})

display(samples.head())


# =========================
# 5. Unir clinical con survival
# =========================

clinical_survival = clinical.merge(
    survival,
    on="sample",
    how="left"
)

print("Clinical + survival:", clinical_survival.shape)
display(clinical_survival.head())


# =========================
# 6. Revisar valores faltantes en expresión
# =========================

print("Valores faltantes totales en X:", X.isna().sum().sum())

missing_by_gene = X.isna().sum(axis=0)
missing_by_sample = X.isna().sum(axis=1)

print("\nGenes con más valores faltantes:")
display(missing_by_gene.sort_values(ascending=False).head())

print("\nMuestras con más valores faltantes:")
display(missing_by_sample.sort_values(ascending=False).head())


# =========================
# 7. Limpiar y filtrar genes
# =========================

X_clean = X.dropna(axis=1)

print("Antes de filtrar:", X_clean.shape)

# Selección de genes con mayor varianza
top_n = 2000

gene_variances = X_clean.var(axis=0)
top_genes = gene_variances.sort_values(ascending=False).head(top_n).index

X_filtered = X_clean[top_genes]

print("Después de seleccionar genes de mayor varianza:", X_filtered.shape)
display(X_filtered.head())


# =========================
# 8. Escalamiento
# =========================

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X_filtered)

X_scaled = pd.DataFrame(
    X_scaled_array,
    index=X_filtered.index,
    columns=X_filtered.columns
)

print("Matriz escalada:", X_scaled.shape)
display(X_scaled.head())

La matriz de expresión génica fue transpuesta para que cada fila representara una muestra/paciente y cada columna representara un gen. Esta estructura es necesaria para aplicar técnicas de aprendizaje no supervisado, como PCA y clustering. Posteriormente, se verificó que las muestras presentes en la matriz de expresión coincidieran con las muestras disponibles en el archivo clínico. Se encontró que las 57 muestras de expresión también estaban presentes en los datos clínicos, mientras que el archivo de supervivencia contenía información para 55 muestras.

Para preparar los datos antes del análisis, se eliminaron genes con valores faltantes y se seleccionaron los genes con mayor varianza. Esta decisión se tomó porque el dataset contiene una cantidad muy alta de genes en comparación con el número de pacientes, por lo que conservar todos los genes podría introducir ruido y dificultar la identificación de patrones relevantes. Finalmente, los datos fueron escalados mediante StandardScaler, con el objetivo de que cada gen tuviera media 0 y desviación estándar 1 antes de aplicar PCA y clustering.